============================================================
PROFESSIONAL SALES DATA ANALYSIS
Python + Pandas + Matplotlib
Generated from: Sales_Data_Analysis_Jupyter.xlsx
============================================================

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------
file_path = "Sales_Data_Analysis_Jupyter.xlsx"

In [ ]:
df = pd.read_excel(file_path, sheet_name="Sales_Data")

In [ ]:
print("=" * 70)
print("SALES DATA ANALYSIS")
print("=" * 70)

In [ ]:
print("\nDataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
print("\nData types:")
print(df.dtypes)

In [ ]:
# ------------------------------------------------------------
# 2. Data quality assessment
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("DATA QUALITY CHECK")
print("=" * 70)

In [ ]:
print("\nMissing values before cleaning:")
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate Order IDs:", df["Order_ID"].duplicated().sum())

In [ ]:
# Remove duplicate orders
df = df.drop_duplicates(subset="Order_ID", keep="first").copy()

In [ ]:
# Fill missing values
df["Region"] = df["Region"].fillna("Unknown")
df["Discount_%"] = df["Discount_%"].fillna(df["Discount_%"].median())
df["Unit_Price"] = df["Unit_Price"].fillna(
    df.groupby("Category")["Unit_Price"].transform("median")
)
df["Unit_Price"] = df["Unit_Price"].fillna(df["Unit_Price"].median())

In [ ]:
# Correct data types
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
df["Units"] = pd.to_numeric(df["Units"], errors="coerce").fillna(0)
df["Discount_%"] = pd.to_numeric(df["Discount_%"], errors="coerce").fillna(0)
df["Unit_Price"] = pd.to_numeric(df["Unit_Price"], errors="coerce").fillna(0)

In [ ]:
# ------------------------------------------------------------
# 3. Feature engineering
# ------------------------------------------------------------
df["Month"] = df["Order_Date"].dt.to_period("M").astype(str)
df["Quarter"] = df["Order_Date"].dt.to_period("Q").astype(str)

In [ ]:
df["Revenue_Calculated"] = (
    df["Units"] * df["Unit_Price"] * (1 - df["Discount_%"] / 100)
)

In [ ]:
df["Gross_Profit_Calculated"] = (
    df["Revenue_Calculated"] - df["Cost"]
)

In [ ]:
df["Profit_Margin_Calculated_%"] = np.where(
    df["Revenue_Calculated"] != 0,
    df["Gross_Profit_Calculated"] / df["Revenue_Calculated"] * 100,
    0
)

In [ ]:
# ------------------------------------------------------------
# 4. Executive KPIs
# ------------------------------------------------------------
total_revenue = df["Revenue_Calculated"].sum()
total_cost = df["Cost"].sum()
total_profit = df["Gross_Profit_Calculated"].sum()
total_units = df["Units"].sum()
total_orders = df["Order_ID"].nunique()

In [ ]:
avg_order_value = total_revenue / total_orders
overall_margin = total_profit / total_revenue * 100

In [ ]:
print("\n" + "=" * 70)
print("EXECUTIVE KPIs")
print("=" * 70)

In [ ]:
print(f"Total Revenue:       {total_revenue:,.2f}")
print(f"Total Cost:          {total_cost:,.2f}")
print(f"Total Profit:        {total_profit:,.2f}")
print(f"Total Units Sold:    {total_units:,.0f}")
print(f"Total Orders:        {total_orders:,}")
print(f"Average Order Value: {avg_order_value:,.2f}")
print(f"Overall Margin:      {overall_margin:.2f}%")

In [ ]:
# ------------------------------------------------------------
# 5. Monthly performance
# ------------------------------------------------------------
monthly = (
    df.groupby("Month")
      .agg(
          Revenue=("Revenue_Calculated", "sum"),
          Cost=("Cost", "sum"),
          Profit=("Gross_Profit_Calculated", "sum"),
          Orders=("Order_ID", "nunique"),
          Units=("Units", "sum")
      )
      .reset_index()
)

In [ ]:
monthly["Profit_Margin_%"] = (
    monthly["Profit"] / monthly["Revenue"] * 100
)

In [ ]:
monthly["Revenue_Growth_%"] = monthly["Revenue"].pct_change() * 100

In [ ]:
print("\nMonthly performance:")
print(monthly.round(2))

In [ ]:
# ------------------------------------------------------------
# 6. Regional analysis
# ------------------------------------------------------------
by_region = (
    df.groupby("Region")
      .agg(
          Revenue=("Revenue_Calculated", "sum"),
          Profit=("Gross_Profit_Calculated", "sum"),
          Orders=("Order_ID", "nunique"),
          Units=("Units", "sum")
      )
      .reset_index()
)

In [ ]:
by_region["Profit_Margin_%"] = (
    by_region["Profit"] / by_region["Revenue"] * 100
)

In [ ]:
by_region["Revenue_Share_%"] = (
    by_region["Revenue"] / total_revenue * 100
)

In [ ]:
by_region = by_region.sort_values("Revenue", ascending=False)

In [ ]:
print("\nRevenue and profit by region:")
print(by_region.round(2))

In [ ]:
# ------------------------------------------------------------
# 7. Product category analysis
# ------------------------------------------------------------
by_category = (
    df.groupby("Category")
      .agg(
          Revenue=("Revenue_Calculated", "sum"),
          Cost=("Cost", "sum"),
          Profit=("Gross_Profit_Calculated", "sum"),
          Units=("Units", "sum"),
          Orders=("Order_ID", "nunique")
      )
      .reset_index()
)

In [ ]:
by_category["Profit_Margin_%"] = (
    by_category["Profit"] / by_category["Revenue"] * 100
)

In [ ]:
by_category["Revenue_Share_%"] = (
    by_category["Revenue"] / total_revenue * 100
)

In [ ]:
by_category = by_category.sort_values("Revenue", ascending=False)

In [ ]:
print("\nCategory performance:")
print(by_category.round(2))

In [ ]:
# ------------------------------------------------------------
# 8. Sales channel analysis
# ------------------------------------------------------------
by_channel = (
    df.groupby("Sales_Channel")
      .agg(
          Revenue=("Revenue_Calculated", "sum"),
          Profit=("Gross_Profit_Calculated", "sum"),
          Orders=("Order_ID", "nunique"),
          Units=("Units", "sum")
      )
      .reset_index()
)

In [ ]:
by_channel["Profit_Margin_%"] = (
    by_channel["Profit"] / by_channel["Revenue"] * 100
)

In [ ]:
by_channel["Revenue_Share_%"] = (
    by_channel["Revenue"] / total_revenue * 100
)

In [ ]:
by_channel = by_channel.sort_values("Revenue", ascending=False)

In [ ]:
print("\nSales channel performance:")
print(by_channel.round(2))

In [ ]:
# ------------------------------------------------------------
# 9. Customer type analysis
# ------------------------------------------------------------
by_customer = (
    df.groupby("Customer_Type")
      .agg(
          Revenue=("Revenue_Calculated", "sum"),
          Profit=("Gross_Profit_Calculated", "sum"),
          Orders=("Order_ID", "nunique"),
          Units=("Units", "sum")
      )
      .reset_index()
)

In [ ]:
by_customer["Profit_Margin_%"] = (
    by_customer["Profit"] / by_customer["Revenue"] * 100
)

In [ ]:
by_customer["Revenue_Share_%"] = (
    by_customer["Revenue"] / total_revenue * 100
)

In [ ]:
print("\nCustomer type performance:")
print(by_customer.round(2))

In [ ]:
# ------------------------------------------------------------
# 10. Discount analysis
# ------------------------------------------------------------
discount_analysis = (
    df.groupby("Discount_%")
      .agg(
          Revenue=("Revenue_Calculated", "sum"),
          Profit=("Gross_Profit_Calculated", "sum"),
          Orders=("Order_ID", "nunique")
      )
      .reset_index()
)

In [ ]:
discount_analysis["Profit_Margin_%"] = (
    discount_analysis["Profit"] / discount_analysis["Revenue"] * 100
)

In [ ]:
print("\nDiscount analysis:")
print(discount_analysis.round(2))

In [ ]:
correlation = df[
    ["Discount_%", "Profit_Margin_Calculated_%"]
].corr().iloc[0, 1]

In [ ]:
print(f"\nDiscount vs Profit Margin correlation: {correlation:.3f}")

In [ ]:
# ------------------------------------------------------------
# 11. Top 10 most profitable orders
# ------------------------------------------------------------
top_10_orders = (
    df.sort_values(
        "Gross_Profit_Calculated",
        ascending=False
    )
    [["Order_ID", "Order_Date", "Region", "Category",
      "Sales_Channel", "Customer_Type", "Units",
      "Discount_%", "Revenue_Calculated",
      "Gross_Profit_Calculated",
      "Profit_Margin_Calculated_%"]]
    .head(10)
)

In [ ]:
print("\nTop 10 profitable orders:")
print(top_10_orders.round(2))

In [ ]:
# ------------------------------------------------------------
# 12. Business insights
# ------------------------------------------------------------
best_month = monthly.loc[monthly["Revenue"].idxmax()]
worst_month = monthly.loc[monthly["Revenue"].idxmin()]
best_region = by_region.iloc[0]
best_category = by_category.iloc[0]
best_margin_category = by_category.loc[
    by_category["Profit_Margin_%"].idxmax()
]
best_channel = by_channel.iloc[0]
best_customer = by_customer.sort_values(
    "Revenue", ascending=False
).iloc[0]

In [ ]:
print("\n" + "=" * 70)
print("KEY BUSINESS INSIGHTS")
print("=" * 70)

In [ ]:
print(
    f"1. Highest-revenue month: {best_month['Month']} "
    f"with revenue of {best_month['Revenue']:,.2f}."
)

In [ ]:
print(
    f"2. Lowest-revenue month: {worst_month['Month']} "
    f"with revenue of {worst_month['Revenue']:,.2f}."
)

In [ ]:
print(
    f"3. Leading region: {best_region['Region']} "
    f"with revenue of {best_region['Revenue']:,.2f} "
    f"({best_region['Revenue_Share_%']:.1f}% of total revenue)."
)

In [ ]:
print(
    f"4. Leading category: {best_category['Category']} "
    f"with revenue of {best_category['Revenue']:,.2f}."
)

In [ ]:
print(
    f"5. Highest-margin category: "
    f"{best_margin_category['Category']} "
    f"with margin of "
    f"{best_margin_category['Profit_Margin_%']:.2f}%."
)

In [ ]:
print(
    f"6. Leading sales channel: {best_channel['Sales_Channel']} "
    f"with revenue of {best_channel['Revenue']:,.2f}."
)

In [ ]:
print(
    f"7. Largest customer segment by revenue: "
    f"{best_customer['Customer_Type']}."
)

In [ ]:
print(
    f"8. Discount/profit-margin correlation: "
    f"{correlation:.3f}. A negative value indicates that "
    f"higher discounts tend to be associated with lower margins."
)

------------------------------------------------------------
13. Visualizations
------------------------------------------------------------

In [ ]:
# Chart 1: Monthly Revenue
plt.figure(figsize=(12, 6))
plt.plot(
    monthly["Month"],
    monthly["Revenue"],
    marker="o"
)
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Revenue by Category
plt.figure(figsize=(10, 6))
plt.bar(
    by_category["Category"],
    by_category["Revenue"]
)
plt.title("Revenue by Product Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: Profit by Region
plt.figure(figsize=(10, 6))
plt.bar(
    by_region["Region"],
    by_region["Profit"]
)
plt.title("Profit by Region")
plt.xlabel("Region")
plt.ylabel("Profit")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 4: Revenue Share by Sales Channel
plt.figure(figsize=(8, 8))
plt.pie(
    by_channel["Revenue"],
    labels=by_channel["Sales_Channel"],
    autopct="%1.1f%%"
)
plt.title("Revenue Share by Sales Channel")
plt.show()

In [ ]:
# ------------------------------------------------------------
# 14. Export cleaned and summarized results
# ------------------------------------------------------------
with pd.ExcelWriter(
    "Sales_Analysis_Output.xlsx",
    engine="openpyxl"
) as writer:
    df.to_excel(writer, sheet_name="Cleaned_Data", index=False)
    monthly.to_excel(writer, sheet_name="Monthly_Performance", index=False)
    by_region.to_excel(writer, sheet_name="Regional_Analysis", index=False)
    by_category.to_excel(writer, sheet_name="Category_Analysis", index=False)
    by_channel.to_excel(writer, sheet_name="Channel_Analysis", index=False)
    by_customer.to_excel(writer, sheet_name="Customer_Analysis", index=False)
    discount_analysis.to_excel(writer, sheet_name="Discount_Analysis", index=False)
    top_10_orders.to_excel(writer, sheet_name="Top_10_Orders", index=False)

In [ ]:
print("\nAnalysis completed successfully.")
print("Output saved as: Sales_Analysis_Output.xlsx")